# Regressão — Preços de Imóveis (Boston Housing)

Demonstração de ponta a ponta da rede neural `Rede`, implementada do zero nesta biblioteca (sem frameworks de ML), aplicada à regressão do **valor mediano de residências**.

**Pipeline:** carregar os dados → dividir em treino/validação/teste → treinar → ajustar a taxa de aprendizado na validação → analisar a curva de aprendizado → avaliar no teste com métricas contínuas (MSE, RMSE e MAE).

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# coloca a raiz do repositorio no path, para importar a biblioteca da api
sys.path.insert(0, os.path.abspath('..'))
from api.main import Rede

## 1. Carregando os dados

O conjunto `regr_housing.csv` tem 506 amostras, 11 atributos numéricos e a coluna alvo contínua `Median_Home_Value`. Diferente da classificação, aqui queremos prever um valor numérico o mais próximo possível do valor real.

In [ ]:
df = pd.read_csv('../datasets/clean/regr_housing.csv')
print('formato:', df.shape)
print('\nEstatísticas da variável alvo (Median_Home_Value):')
print(df['Median_Home_Value'].describe())
df.head()

## 2. Preparação dos dados

Assim como na classificação binária, preparamos os dados convertendo para arrays NumPy e particionando em **treino / validação / teste (60/20/20)** para garantir uma avaliação justa do modelo e do ajuste de hiperparâmetros.

In [ ]:
# X = atributos, y = alvo continuo
X = df.drop(columns=['Median_Home_Value']).to_numpy()
y = df[['Median_Home_Value']].to_numpy().astype(float)

# divisao treino/validacao/teste (60/20/20) com embaralhamento reprodutivel
rng = np.random.default_rng(42)
indices = rng.permutation(len(X))
n_train = int(0.60 * len(X))
n_val = int(0.20 * len(X))
test_idx = indices[n_train + n_val:]
train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

print('treino:', X_train.shape, '| validacao:', X_val.shape, '| teste:', X_test.shape)

## 3. Funções auxiliares: predição e métricas

Para a regressão, usamos o `feedforward` para obter a predição contínua direta. As métricas avaliadas incluem o **MSE** (Erro Quadrático Médio), **RMSE** (Raiz do Erro Quadrático Médio) e **MAE** (Erro Absoluto Médio), implementadas utilizando apenas `numpy`.

In [ ]:
# valor previsto para cada amostra
def predict_values(rede, X_matrix):
    predictions = []
    for i in range(len(X_matrix)):
        x = np.append(X_matrix[i], 1.0)  # acrescenta o slot do bias
        prediction, _ = rede.feedforward(x, np.zeros(1))  # y ficticio
        predictions.append(prediction[0])
    return np.array(predictions)

# metricas de avaliacao para regressao
def regression_metrics(y_true, y_pred):
    y_true = y_true.ravel()
    y_pred = y_pred.ravel()
    mse = np.mean((y_true - y_pred)**2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(y_true - y_pred))
    return {'mse': mse, 'rmse': rmse, 'mae': mae}

## 4. Treino e ajuste da taxa de aprendizado (validação)

Arquitetura: **11 atributos → 16 neurônios ReLU (camada oculta) → 1 neurônio linear (saída = valor predito)**, custo erro quadrático médio (MSE).

Neste caso, diferentemente da classificação, a métrica de sucesso é ter o **menor MSE possível** no conjunto de validação.

In [ ]:
# cria e configura a rede
def build_network(learning_rate):
    rede = Rede(learning_rate, atributes=X_train, labels=y_train)
    rede.weights_initialization_mode = 'random'
    rede.create_initial_layer(16, 'relu')        # camada oculta
    rede.create_hidden_layer(1, 'linear')        # camada de saida (ativacao linear para regressao)
    rede.set_cost_function('mean_squared_error')
    return rede

# ajuste da taxa de aprendizado usando o conjunto de validacao
# Taxas em regressao costumam ser menores para evitar explosoes de gradiente
candidate_learning_rates = [0.0001, 0.001, 0.01]
results = []
for lr in candidate_learning_rates:
    np.random.seed(0)
    candidate = build_network(lr)
    history = candidate.train(epochs=60)
    val_pred = predict_values(candidate, X_val)
    val_mse = regression_metrics(y_val, val_pred)['mse']
    results.append({'lr': lr, 'val_mse': val_mse, 'model': candidate, 'history': history})
    print(f'lr={lr:<7} MSE na validacao = {val_mse:.4f}')

# Busca o modelo com o MENOR MSE
best = min(results, key=lambda r: r['val_mse'])
best_model = best['model']
best_history = best['history']
print()
print(f"melhor taxa de aprendizado: {best['lr']} (validacao MSE = {best['val_mse']:.4f})")

## 5. Curva de aprendizado

O progresso da função de custo (MSE) através das épocas. Como lidamos com erros não-limitados (diferente das probabilidades na sigmoide), a curva pode mostrar quedas acentuadas nas primeiras iterações.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(best_history)
plt.title('Curva de aprendizado (melhor modelo) - MSE medio por epoca')
plt.xlabel('epoca')
plt.ylabel('perda (MSE) media')
plt.grid(True, alpha=0.3)
plt.show()

## 6. Avaliação final no conjunto de teste

Finalmente testamos a capacidade de generalização do modelo nos dados isolados de teste. Ao invés da matriz de confusão, para regressão é muito útil visualizar um gráfico de dispersão com os valores **Reais vs. Previstos**. Quanto mais próximos da reta identidade (y=x), melhor.

In [ ]:
test_pred = predict_values(best_model, X_test)
m = regression_metrics(y_test, test_pred)

print('Desempenho no conjunto de teste:')
print(f"  MSE  (Erro Quadrático Médio):        {m['mse']:.4f}")
print(f"  RMSE (Raiz do Erro Quad. Médio):   {m['rmse']:.4f}")
print(f"  MAE  (Erro Absoluto Médio):          {m['mae']:.4f}")

In [ ]:
# Gráfico de dispersão: Valores Reais vs Previstos
plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_pred, alpha=0.7, edgecolors='w', s=60, color='teal')

# Linha de referencia ideal (predicao exata)
min_val = min(y_test.min(), test_pred.min())
max_val = max(y_test.max(), test_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Predição Ideal (y=x)')

plt.xlabel('Valores Reais')
plt.ylabel('Valores Previstos')
plt.title('Valores Reais vs. Previstos - Teste')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 7. Conclusão

A rede neural foi capaz de identificar os padrões não-lineares nas métricas imobiliárias e fornecer estimativas numéricas de valor.

Pontos importantes na Regressão:

- **Função de Ativação**: A camada de saída utiliza a ativação `linear`, garantindo que não estamos limitando o limite do output artificialmente a domínios como (0, 1) ou (-1, 1).
- **Otimização**: Escolher a taxa de aprendizado em problemas de regressão deve ser feito de forma conservadora. Um LR alto pode causar explosão do gradiente instantaneamente devido aos valores maiores presentes no cálculo do gradiente em relação à função MSE.
- **Interpretação**: O **MAE** nos diz diretamente, em média, a quantos "milhares de dólares" nossa predição está errando, sendo uma métrica muito melhor para percepção de negócio do que o MSE puro.